# Exploratory Data Analysis
## Ethereum Phishing Transaction Network

This notebook explores the `MulDiGraph.pkl` dataset and the extracted tabular features.

**Sections:**
1. Load graph + basic stats
2. Label distribution
3. Degree analysis
4. Transaction amount analysis
5. Timestamp analysis
6. Load processed feature table
7. Correlation heatmap
8. Phishing vs Normal feature comparison

In [ ]:
import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns

%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')

ROOT = Path('.').resolve().parent  # notebooks/ → root
sys.path.insert(0, str(ROOT))
print('Root:', ROOT)

## 1. Load Graph

In [ ]:
GRAPH_CANDIDATES = [
    ROOT / 'data' / 'raw' / 'MulDiGraph.pkl',
    ROOT / 'Ethereum Phishing Transaction Network' / 'MulDiGraph.pkl',
]

graph_path = next((p for p in GRAPH_CANDIDATES if p.exists()), None)
if graph_path is None:
    raise FileNotFoundError('MulDiGraph.pkl not found.')

print(f'Loading from: {graph_path}')
with open(graph_path, 'rb') as f:
    G = pickle.load(f)

print(nx.info(G))
print(f'Type: {type(G).__name__}')

## 2. Label Distribution

In [ ]:
labels = [d.get('isp', 0) for _, d in G.nodes(data=True)]
n_phishing = sum(labels)
n_normal = len(labels) - n_phishing

print(f'Total nodes   : {len(labels):,}')
print(f'Phishing (isp=1): {n_phishing:,}  ({n_phishing/len(labels)*100:.4f}%)')
print(f'Normal   (isp=0): {n_normal:,}  ({n_normal/len(labels)*100:.4f}%)')

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(['Normal', 'Phishing'], [n_normal, n_phishing], color=['#21c354', '#ff4b4b'])
ax.set_title('Node Label Distribution (all nodes)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 3. Degree Analysis (labeled nodes only)

In [ ]:
phishing_nodes = [n for n, d in G.nodes(data=True) if d.get('isp', 0) == 1]
import random; random.seed(42)
sample_normal  = random.sample([n for n, d in G.nodes(data=True) if d.get('isp', 0) == 0],
                                min(5000, n_normal))

def get_degrees(nodes):
    return [G.in_degree(n) for n in nodes], [G.out_degree(n) for n in nodes]

phi_in, phi_out = get_degrees(phishing_nodes)
nor_in, nor_out = get_degrees(sample_normal)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, phi_vals, nor_vals, title in zip(axes,
    [phi_in, phi_out], [nor_in, nor_out],
    ['In-Degree', 'Out-Degree']):
    ax.hist(np.clip(nor_vals, 0, 50), bins=50, alpha=0.6, label='Normal', color='#21c354')
    ax.hist(np.clip(phi_vals, 0, 50), bins=50, alpha=0.7, label='Phishing', color='#ff4b4b')
    ax.set_title(f'{title} Distribution (clipped at 50)')
    ax.set_xlabel(title)
    ax.legend()
plt.tight_layout()
plt.show()

## 4. Transaction Amount Sampling

In [ ]:
# Sample up to 50,000 edge amounts for inspection
amounts = []
for i, (u, v, d) in enumerate(G.edges(data=True)):
    if i > 50000: break
    a = d.get('amount', 0)
    if a and a > 0:
        amounts.append(float(a))

print(f'Sample edges: {len(amounts):,}')
s = pd.Series(amounts)
print(s.describe())

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(np.log1p(s), bins=80, color='steelblue', edgecolor='none')
ax.set_title('Edge amounts – log1p distribution')
ax.set_xlabel('log1p(amount)')
plt.tight_layout()
plt.show()

## 5. Load Processed Feature Table

In [ ]:
train_path = ROOT / 'data' / 'processed' / 'train.csv'
test_path  = ROOT / 'data' / 'processed' / 'test.csv'

if not train_path.exists():
    print('Run ml/preprocess.py first.')
else:
    train_df = pd.read_csv(train_path)
    test_df  = pd.read_csv(test_path)
    df = pd.concat([train_df, test_df], ignore_index=True)
    print(f'Full dataset: {df.shape}  |  Phishing: {df.label.sum()} ({df.label.mean()*100:.1f}%)')
    df.head(3)

## 6. Correlation Heatmap

In [ ]:
if 'df' in dir():
    feat_cols = [c for c in df.columns if c not in ('label', 'node_id')]
    corr = df[feat_cols + ['label']].corr()

    fig, ax = plt.subplots(figsize=(14, 10))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
                linewidths=0.3, ax=ax, annot_kws={'size': 7})
    ax.set_title('Feature Correlation Heatmap')
    plt.tight_layout()
    plt.show()

## 7. Phishing vs Normal – Feature Box Plots

In [ ]:
if 'df' in dir():
    top_features = [
        'in_degree', 'out_degree',
        'total_received', 'total_sent',
        'avg_in_amount', 'avg_out_amount',
        'in_out_ratio', 'sent_received_ratio',
    ]
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()

    for ax, feat in zip(axes, top_features):
        tmp = df[[feat, 'label']].copy()
        tmp['Group'] = tmp['label'].map({0: 'Normal', 1: 'Phishing'})
        tmp[feat] = np.log1p(tmp[feat].clip(lower=0))
        sns.boxplot(data=tmp, x='Group', y=feat, ax=ax,
                    palette={'Normal': '#21c354', 'Phishing': '#ff4b4b'},
                    showfliers=False)
        ax.set_title(f'log1p({feat})')
        ax.set_xlabel('')

    plt.suptitle('Feature Distributions: Phishing vs Normal', y=1.01)
    plt.tight_layout()
    plt.show()